# M4: Final Hybrid Architecture Validation (Frozen-MiniLM Stage)

This is the accepted BPR + recency + 50/50 fusion pipeline **before** M2.2 fine-tuned MiniLM embeddings replaced the frozen encoder in the final M5 refit. This notebook was originally the frozen encoder validation baseline; the fine-tuned validation comparison lives in M2.2. No review, SASRec, or XGBoost models are included.


## 1. Imports and fixed configuration

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path('.')

TRAIN_PATH = DATA_DIR / 'train_interactions.parquet'
VALID_PATH = DATA_DIR / 'valid_interactions.parquet'
TEST_PATH = DATA_DIR / 'test_interactions.parquet'

BPR_CHECKPOINT = DATA_DIR / 'bpr_m1a_all_interactions.pt'
M2_EMBED_PATH = DATA_DIR / 'metadata_minilm_embeddings.npy'
M2_INDEX_PATH = DATA_DIR / 'metadata_embedding_index.parquet'

CANDIDATE_K = 500
RECENT_N = 2
RHO = 0.60
ALPHA_BPR = 0.50
ALPHA_META = 1.0 - ALPHA_BPR
BATCH_SIZE = 96

print('Final M4 configuration')
print('Candidate K      :', CANDIDATE_K)
print('Recent N         :', RECENT_N)
print('Recency rho      :', RHO)
print('BPR weight       :', ALPHA_BPR)
print('Metadata weight  :', ALPHA_META)
print('Batch size       :', BATCH_SIZE)

## 2. Load data and expensive saved representations

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
valid = pd.read_parquet(VALID_PATH)
test = pd.read_parquet(TEST_PATH)

metadata_embeddings = np.load(M2_EMBED_PATH).astype(np.float32)
metadata_index = pd.read_parquet(M2_INDEX_PATH)

# Normalize item metadata embeddings again for safety.
meta_norms = np.linalg.norm(metadata_embeddings, axis=1, keepdims=True)
metadata_embeddings = metadata_embeddings / np.clip(meta_norms, 1e-12, None)

state_dict = torch.load(BPR_CHECKPOINT, map_location='cpu')
bpr_user_embeddings = (
    state_dict['user_embedding.weight'].cpu().numpy().astype(np.float32)
)
bpr_item_embeddings = (
    state_dict['item_embedding.weight'].cpu().numpy().astype(np.float32)
)

print('Train:', train.shape)
print('Valid:', valid.shape)
print('Test :', test.shape)
print('Metadata embeddings:', metadata_embeddings.shape)
print('Metadata index     :', metadata_index.shape)
print('BPR user embeddings:', bpr_user_embeddings.shape)
print('BPR item embeddings:', bpr_item_embeddings.shape)

assert valid['user_id'].nunique() == len(valid)
assert test['user_id'].nunique() == len(test)

## 3. Reconstruct the exact M1a mappings

The BPR checkpoint was trained with **first-appearance ordering**:

- users: `TRAIN.user_id.unique()`
- items: `TRAIN -> VALID -> TEST`, followed by `.unique()`

Do not sort these IDs.

In [ ]:
all_users = train['user_id'].unique()
all_items = pd.concat([
    train['parent_asin'],
    valid['parent_asin'],
    test['parent_asin'],
]).unique()

user_to_idx = {user: idx for idx, user in enumerate(all_users)}
item_to_idx = {item: idx for idx, item in enumerate(all_items)}
idx_to_item = np.asarray(all_items)

NUM_USERS = len(all_users)
NUM_ITEMS = len(all_items)

assert NUM_USERS == bpr_user_embeddings.shape[0]
assert NUM_ITEMS == bpr_item_embeddings.shape[0]

print('Users:', NUM_USERS)
print('Items:', NUM_ITEMS)
print('BPR checkpoint mapping aligned.')

## 4. Build TRAIN history, seen-item masks, and BPR retrieval catalog

In [ ]:
train_indexed = train[['user_id', 'parent_asin', 'timestamp']].copy()
train_indexed['user_idx'] = (
    train_indexed['user_id'].map(user_to_idx).astype(np.int32)
)
train_indexed['item_idx'] = (
    train_indexed['parent_asin'].map(item_to_idx).astype(np.int32)
)

user_seen = (
    train_indexed.groupby('user_idx')['item_idx'].apply(set).to_dict()
)

# Only TRAIN-observed items have trained BPR item embeddings.
bpr_catalog_idx = np.array(
    sorted({item_to_idx[item] for item in train['parent_asin'].unique()}),
    dtype=np.int32,
)

global_to_bpr_pos = np.full(NUM_ITEMS, -1, dtype=np.int32)
global_to_bpr_pos[bpr_catalog_idx] = np.arange(
    len(bpr_catalog_idx), dtype=np.int32
)

bpr_trained_mask = np.zeros(NUM_ITEMS, dtype=bool)
bpr_trained_mask[bpr_catalog_idx] = True

print('BPR retrieval catalog:', len(bpr_catalog_idx))
print('TRAIN interactions    :', len(train_indexed))

## 5. Align MiniLM metadata embeddings to global item IDs

In [ ]:
asin_to_meta_idx = {
    asin: idx for idx, asin in enumerate(metadata_index['parent_asin'])
}

global_to_meta_idx = np.array(
    [asin_to_meta_idx.get(asin, -1) for asin in all_items],
    dtype=np.int32,
)

missing_metadata = int((global_to_meta_idx < 0).sum())
print('Global items missing metadata:', missing_metadata)
assert missing_metadata == 0

# Metadata vectors are now indexed by the same global item index used by BPR.
global_metadata_embeddings = metadata_embeddings[global_to_meta_idx]
metadata_catalog_embeddings = global_metadata_embeddings

train_indexed['meta_idx'] = (
    train_indexed['parent_asin'].map(asin_to_meta_idx).astype(np.int32)
)

print('Metadata retrieval catalog:', len(metadata_catalog_embeddings))

## 6. Build all-history and recent-2 metadata user profiles

The all-history profile captures long-term preference. The recent profile uses only the latest two TRAIN interactions. Both are normalized before blending.

In [ ]:
def l2_normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    ok = norms[:, 0] > 1e-12
    matrix[ok] /= norms[ok]
    return matrix, ok


# Long-term profile from all TRAIN interactions.
all_history_matrix = csr_matrix(
    (
        np.ones(len(train_indexed), dtype=np.float32),
        (train_indexed['user_idx'], train_indexed['meta_idx']),
    ),
    shape=(NUM_USERS, len(metadata_index)),
    dtype=np.float32,
)

all_history_profiles = np.asarray(
    all_history_matrix @ metadata_embeddings,
    dtype=np.float32,
)
all_history_profiles, has_all_profile = l2_normalize_rows(all_history_profiles)


# Recent profile from each user's latest RECENT_N TRAIN interactions.
chronological = train_indexed.sort_values(
    ['user_idx', 'timestamp'], kind='mergesort'
)
recent_rows = (
    chronological
    .groupby('user_idx', sort=False, group_keys=False)
    .tail(RECENT_N)
)

recent_matrix = csr_matrix(
    (
        np.ones(len(recent_rows), dtype=np.float32),
        (recent_rows['user_idx'], recent_rows['meta_idx']),
    ),
    shape=(NUM_USERS, len(metadata_index)),
    dtype=np.float32,
)

recent_profiles = np.asarray(
    recent_matrix @ metadata_embeddings,
    dtype=np.float32,
)
recent_profiles, has_recent_profile = l2_normalize_rows(recent_profiles)

print('All-history profiles:', all_history_profiles.shape)
print('Recent profiles     :', recent_profiles.shape)
print('Users with all-history profile:', int(has_all_profile.sum()))
print('Users with recent profile     :', int(has_recent_profile.sum()))

## 7. Prepare full VALID users and targets

In [ ]:
valid_user_idx = np.array(
    [user_to_idx[user] for user in valid['user_id'].to_numpy()],
    dtype=np.int32,
)

valid_target_idx = np.array(
    [item_to_idx[item] for item in valid['parent_asin'].to_numpy()],
    dtype=np.int32,
)

NUM_VALID = len(valid_user_idx)

cold_valid_targets = ~bpr_trained_mask[valid_target_idx]

print('Validation users             :', NUM_VALID)
print('Targets unseen in BPR TRAIN :', int(cold_valid_targets.sum()))

## 8. Ranking helpers

In [ ]:
def topk_sorted(scores, k):
    '''Return sorted top-k column positions and their scores.'''
    local = np.argpartition(scores, -k, axis=1)[:, -k:]
    top_scores = np.take_along_axis(scores, local, axis=1)
    order = np.argsort(top_scores, axis=1)[:, ::-1]
    local = np.take_along_axis(local, order, axis=1)
    top_scores = np.take_along_axis(top_scores, order, axis=1)
    return local, top_scores


def safe_zscore_1d(values):
    '''Z-score finite values only; leave unavailable values as NaN.'''
    values = np.asarray(values, dtype=np.float32)
    finite = np.isfinite(values)
    out = np.full(values.shape, np.nan, dtype=np.float32)

    if finite.sum() == 0:
        return out
    if finite.sum() == 1:
        out[finite] = 0.0
        return out

    mean = values[finite].mean()
    std = values[finite].std()
    if std < 1e-8:
        out[finite] = 0.0
    else:
        out[finite] = (values[finite] - mean) / std
    return out


def metrics_from_top10(top10, target_idx):
    '''Return hit, reciprocal rank, and NDCG for one user.'''
    match = np.where(top10 == int(target_idx))[0]
    if len(match) == 0:
        return 0, 0.0, 0.0
    rank = int(match[0]) + 1
    return 1, 1.0 / rank, 1.0 / np.log2(rank + 1)

## 9. Final M4 validation

For every validation user:

1. BPR retrieves top-500 from TRAIN-observed items.
2. Recency-aware metadata retrieves top-500 from the full metadata catalog.
3. The two lists are unioned and deduplicated.
4. BPR and metadata scores are z-normalized **within that user's union**.
5. Metadata-only cold items receive a conservative missing-BPR value below the user's lowest finite BPR z-score.
6. Final score is `0.50 * BPR_z + 0.50 * Metadata_z`.
7. Top-10 is evaluated with HR@10, MRR@10, and NDCG@10.

No candidate matrices are saved.

In [ ]:
bpr_catalog_embeddings = bpr_item_embeddings[bpr_catalog_idx]

# Retrieval diagnostics.
bpr_hits_500 = 0
meta_hits_500 = 0
union_hits = 0
meta_rescued = 0
both_retrieve = 0
neither_retrieve = 0

# Standalone BPR top-10 sanity check.
bpr_hr10 = 0
bpr_mrr10 = 0.0
bpr_ndcg10 = 0.0

# Standalone recency-metadata top-10 diagnostic.
meta_hr10 = 0
meta_mrr10 = 0.0
meta_ndcg10 = 0.0

# Final hybrid metrics.
hybrid_hr10 = 0
hybrid_mrr10 = 0.0
hybrid_ndcg10 = 0.0

union_sizes = []

for start in tqdm(
    range(0, NUM_VALID, BATCH_SIZE),
    desc='Final M4 validation',
):
    end = min(start + BATCH_SIZE, NUM_VALID)
    batch_users = valid_user_idx[start:end]
    batch_targets = valid_target_idx[start:end]

    # ------------------------------------------------------------
    # A. BPR scores over TRAIN-observed catalog
    # ------------------------------------------------------------
    bpr_scores = (
        bpr_user_embeddings[batch_users]
        @ bpr_catalog_embeddings.T
    )

    # ------------------------------------------------------------
    # B. Metadata score components over full metadata catalog
    # ------------------------------------------------------------
    all_meta_scores = (
        all_history_profiles[batch_users]
        @ metadata_catalog_embeddings.T
    )
    recent_meta_scores = (
        recent_profiles[batch_users]
        @ metadata_catalog_embeddings.T
    )

    # Mask TRAIN-seen items from both retrieval branches.
    for row, user_idx in enumerate(batch_users):
        seen = user_seen.get(int(user_idx), set())
        if not seen:
            continue

        seen_global = np.fromiter(seen, dtype=np.int32)

        bpr_pos = global_to_bpr_pos[seen_global]
        bpr_pos = bpr_pos[bpr_pos >= 0]
        if len(bpr_pos):
            bpr_scores[row, bpr_pos] = -np.inf

        all_meta_scores[row, seen_global] = -np.inf
        recent_meta_scores[row, seen_global] = -np.inf

    # Accepted M2.6 recency blend.
    meta_scores = (
        (1.0 - RHO) * all_meta_scores
        + RHO * recent_meta_scores
    )

    # ------------------------------------------------------------
    # C. Top-500 from each retriever
    # ------------------------------------------------------------
    bpr_local, _ = topk_sorted(bpr_scores, CANDIDATE_K)
    bpr_candidates = bpr_catalog_idx[bpr_local]

    meta_local, _ = topk_sorted(meta_scores, CANDIDATE_K)
    # metadata catalog is in global item order
    meta_candidates = meta_local.astype(np.int32)

    # ------------------------------------------------------------
    # D. Per-user union + final 50/50 reranking
    # ------------------------------------------------------------
    for r, (user_idx, target_idx) in enumerate(zip(batch_users, batch_targets)):
        bpr_list = bpr_candidates[r]
        meta_list = meta_candidates[r]

        bpr_hit500 = bool(np.any(bpr_list == int(target_idx)))
        meta_hit500 = bool(np.any(meta_list == int(target_idx)))

        bpr_hits_500 += int(bpr_hit500)
        meta_hits_500 += int(meta_hit500)
        union_hits += int(bpr_hit500 or meta_hit500)
        meta_rescued += int((not bpr_hit500) and meta_hit500)
        both_retrieve += int(bpr_hit500 and meta_hit500)
        neither_retrieve += int((not bpr_hit500) and (not meta_hit500))

        # Individual retriever top-10 diagnostics.
        h, rr, dcg = metrics_from_top10(bpr_list[:10], target_idx)
        bpr_hr10 += h
        bpr_mrr10 += rr
        bpr_ndcg10 += dcg

        h, rr, dcg = metrics_from_top10(meta_list[:10], target_idx)
        meta_hr10 += h
        meta_mrr10 += rr
        meta_ndcg10 += dcg

        # Union and deduplicate. Ordering is irrelevant because we rerank below.
        union = np.unique(np.concatenate([bpr_list, meta_list])).astype(np.int32)
        union_sizes.append(len(union))

        # BPR scores for union items. Cold metadata-only items have no trained BPR score.
        bpr_union_scores = np.full(len(union), np.nan, dtype=np.float32)
        trained = bpr_trained_mask[union]
        if trained.any():
            positions = global_to_bpr_pos[union[trained]]
            bpr_union_scores[trained] = bpr_scores[r, positions]

        # Metadata scores exist for the full catalog.
        meta_union_scores = meta_scores[r, union].astype(np.float32)

        bpr_z = safe_zscore_1d(bpr_union_scores)
        meta_z = safe_zscore_1d(meta_union_scores)

        # Conservative fallback for truly cold items with no trained BPR embedding.
        finite_bpr = np.isfinite(bpr_z)
        if finite_bpr.any():
            bpr_fill = float(bpr_z[finite_bpr].min()) - 1.0
        else:
            bpr_fill = 0.0
        bpr_z = np.where(finite_bpr, bpr_z, bpr_fill)

        # Metadata should be finite for every union item.
        meta_z = np.nan_to_num(meta_z, nan=0.0)

        final_scores = (
            ALPHA_BPR * bpr_z
            + ALPHA_META * meta_z
        )

        top10_local = np.argsort(final_scores)[-10:][::-1]
        final_top10 = union[top10_local]

        h, rr, dcg = metrics_from_top10(final_top10, target_idx)
        hybrid_hr10 += h
        hybrid_mrr10 += rr
        hybrid_ndcg10 += dcg

## 10. Final metrics and sanity checks

In [ ]:
results = {
    'BPR Recall@500': bpr_hits_500 / NUM_VALID,
    'Metadata Recall@500': meta_hits_500 / NUM_VALID,
    'Union Candidate Recall': union_hits / NUM_VALID,
    'Metadata-rescued targets': int(meta_rescued),
    'Both retrieve target': int(both_retrieve),
    'Neither retrieves target': int(neither_retrieve),
    'Mean union size': float(np.mean(union_sizes)),
    'BPR HR@10': bpr_hr10 / NUM_VALID,
    'BPR MRR@10': bpr_mrr10 / NUM_VALID,
    'BPR NDCG@10': bpr_ndcg10 / NUM_VALID,
    'Metadata HR@10': meta_hr10 / NUM_VALID,
    'Metadata MRR@10': meta_mrr10 / NUM_VALID,
    'Metadata NDCG@10': meta_ndcg10 / NUM_VALID,
    'Final Hybrid HR@10': hybrid_hr10 / NUM_VALID,
    'Final Hybrid MRR@10': hybrid_mrr10 / NUM_VALID,
    'Final Hybrid NDCG@10': hybrid_ndcg10 / NUM_VALID,
}

results

In [ ]:
summary = pd.DataFrame([
    {
        'Model': 'BPR',
        'Candidate Recall': results['BPR Recall@500'],
        'HR@10': results['BPR HR@10'],
        'MRR@10': results['BPR MRR@10'],
        'NDCG@10': results['BPR NDCG@10'],
    },
    {
        'Model': 'Recency Metadata',
        'Candidate Recall': results['Metadata Recall@500'],
        'HR@10': results['Metadata HR@10'],
        'MRR@10': results['Metadata MRR@10'],
        'NDCG@10': results['Metadata NDCG@10'],
    },
    {
        'Model': 'Final Hybrid',
        'Candidate Recall': results['Union Candidate Recall'],
        'HR@10': results['Final Hybrid HR@10'],
        'MRR@10': results['Final Hybrid MRR@10'],
        'NDCG@10': results['Final Hybrid NDCG@10'],
    },
])

summary

In [ ]:
# The BPR row should approximately reproduce the established M1a baseline.
EXPECTED_BPR_HR10 = 0.04597834575040628
EXPECTED_BPR_NDCG10 = 0.023712798891119248

print('BPR HR@10 difference from established baseline  :',
      results['BPR HR@10'] - EXPECTED_BPR_HR10)
print('BPR NDCG@10 difference from established baseline:',
      results['BPR NDCG@10'] - EXPECTED_BPR_NDCG10)

assert abs(results['BPR HR@10'] - EXPECTED_BPR_HR10) < 1e-8
assert abs(results['BPR NDCG@10'] - EXPECTED_BPR_NDCG10) < 1e-8

print()
print('Sanity check passed: BPR baseline reproduced exactly.')

## 11. Improvement over the BPR baseline

In [ ]:
hr_rel = (
    results['Final Hybrid HR@10'] / results['BPR HR@10'] - 1.0
) * 100.0
ndcg_rel = (
    results['Final Hybrid NDCG@10'] / results['BPR NDCG@10'] - 1.0
) * 100.0
recall_rel = (
    results['Union Candidate Recall'] / results['BPR Recall@500'] - 1.0
) * 100.0

improvement = pd.DataFrame([
    {'Metric': 'Candidate Recall', 'Relative improvement vs BPR (%)': recall_rel},
    {'Metric': 'HR@10', 'Relative improvement vs BPR (%)': hr_rel},
    {'Metric': 'NDCG@10', 'Relative improvement vs BPR (%)': ndcg_rel},
])

improvement

## 12. Save only the final validation summary

No candidate matrices, metadata profiles, or intermediate score arrays are saved. The expensive reusable artifacts remain the existing BPR checkpoint and MiniLM embedding matrix.

In [ ]:
FINAL_RESULTS_PATH = DATA_DIR / 'm4_final_validation_summary.json'

payload = {
    'configuration': {
        'candidate_k_per_retriever': CANDIDATE_K,
        'recent_n': RECENT_N,
        'rho': RHO,
        'bpr_weight': ALPHA_BPR,
        'metadata_weight': ALPHA_META,
    },
    'metrics': {k: float(v) if isinstance(v, (np.floating, float)) else int(v)
                if isinstance(v, (np.integer, int)) else v
                for k, v in results.items()},
    'relative_improvement_vs_bpr_pct': {
        'Candidate Recall': float(recall_rel),
        'HR@10': float(hr_rel),
        'NDCG@10': float(ndcg_rel),
    },
}

with open(FINAL_RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)

print('Saved:', FINAL_RESULTS_PATH)

## Final accepted M4 architecture

`BPR top-500 + recency-aware MiniLM metadata top-500 -> union -> 50/50 normalized linear reranking -> top-10`

This notebook intentionally contains no M3 review branch, SASRec, or XGBRanker. The next stage is the **one-time TEST evaluation**, which should use the frozen architecture and hyperparameters selected during validation.